# How a historical rating gets made

The daily run scores today. This notebook scores a **Monday in 2018**, and the
only thing that differs is where the articles came from — everything below
`snapshot_select` is `pipeline._process_country` with `as_of` pinned. If the
backfill had its own scoring path the series would be measuring the backfill.

So the interesting parts are the two ends: the line that keeps the future out,
and the anonymizer that lets the scorer judge a country it cannot name.

**Why anonymize at all.** A model asked to rate Türkiye in 2018 may simply
*remember* 2018. Masking removes the identity and keeps every number, so the
score has to come from the evidence in front of it rather than from what the
model already knows about the country — and a 2016 backfill and tomorrow's live
run become the same instrument.

**Two layers and one gate**, in the order they run:

| | What it is | What it catches | What it cannot |
|---|---|---|---|
| **1** | `gazetteer` — a hand-written list, deterministic and offline. Two passes: `mask()` turns the scored country into its roles, then `mask_foreign()` flattens every *other* roster country to "another country" | Names, demonyms, currencies, capitals, central banks, statistics offices, regions — and identification by elimination | Anything nobody wrote down |
| **2** | `rewrite` — two model passes, one over digests + headlines, one over the full bodies the scorer reads end to end | This year's finance minister, this year's party, a named law, a named crisis | Anything the model misses |
| **gate** | `assert_clean` — scans the whole outbound payload, keys as well as values, against the whole roster | A leak either layer missed | Nothing. It raises. |

There is also a **meter**, `probe`, which is not a layer: it asks a cheap model
to name the country and records the answer without ever acting on it.

The gate is fatal on purpose. A masked snapshot that names its country is not a
degraded row, it is a mislabelled one, and in a ten-year series it would look
exactly like a sound one forever after.

**This notebook writes nothing.** No `upsert_snapshot`, no `upsert_probe_result`,
no ledger row. It walks the path and stops before each write.

In [ ]:
import datetime
import json
import logging
import os
import pathlib
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display

# Repo root = the folder holding backend/main.py, so this works from any cwd.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

# force=True is not optional: Jupyter installs its own root handler, so a plain
# basicConfig() is silently a no-op and no pipeline logs ever appear.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s",
                    stream=sys.stdout, force=True)
# httpx logs a line per request, per redirect and per OAuth bounce - about fifty
# of them for one country's news fetch, none about that country's risk. The
# pipeline's own loggers are the point of setting INFO in the first place.
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

from backend.data_upsert import store
from backend.llm import client as ai_client
from backend.llm import gazetteer, probe, rewrite, usage
from backend.llm import langchain_llm
from backend.news_fetching import article_enrichment, article_ranking, core, snapshot_select
from backend.util import config, constants

print("keys")
for key in ("OPENAI_API_KEY", "DATABASE_URL"):
    print(f"  {key:<18} {'set' if os.getenv(key) else 'MISSING'}")

print("\nmasking versions — every one of these is stamped into a masked row's manifest")
print(f"  mask_map_version   {gazetteer.MASK_MAP_VERSION:<10} hand-maintained; the map's DATA")
print(f"  gazetteer_version  {gazetteer.GAZETTEER_VERSION:<10} sha256 of gazetteer.py; the map's CODE")
print(f"  sweep_version      {rewrite.SWEEP_VERSION:<10} sha256 of the digest-sweep prompt")
print(f"  rewrite_version    {rewrite.REWRITE_VERSION:<10} sha256 of the body-rewrite prompt + schema")
print(f"  probe_version      {probe.PROBE_VERSION:<10} sha256 of the probe prompt")
print(f"\ndigest/sweep/probe model  {ai_client.DIGEST_MODEL_NAME}")
print(f"roster {len(gazetteer.DEFAULT_ROSTER)} countries, of which "
      f"{len(config.PILOT_ROSTER)} are in the pilot: {', '.join(config.PILOT_ROSTER)}")

In [ ]:
# --- display kit ---------------------------------------------------------
# Trimmed from country_rating_walkthrough.ipynb's chart kit — same CSS, same
# no-plotting-dependency rule (matplotlib is not in the venv and the kernel is
# deliberately out of requirements.txt). Inline SVG and HTML render in
# JupyterLab, VS Code and GitHub's viewer alike.

_CSS = """<style>
.viz{color-scheme:light;--s:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--rule:#e5e4e1;
--c1:#2a78d6;--c2:#eb6834;--c3:#1baf7a;--c4:#eda100;--c0:#6f6e6a;--crit:#d03b3b;
font:12px/1.45 ui-sans-serif,-apple-system,"Segoe UI",sans-serif;background:var(--s);
border:1px solid var(--rule);border-radius:6px;padding:14px 16px;margin:4px 0;max-width:960px}
@media (prefers-color-scheme:dark){.viz{color-scheme:dark;--s:#1a1a19;--ink:#fff;
--ink2:#c3c2b7;--rule:#3a3a37;--c1:#3987e5;--c2:#d95926;--c3:#199e70;--c4:#c98500;
--c0:#8f8e88}}
.viz h4{margin:0;font-size:13px;font-weight:600;color:var(--ink)}
.viz .sub{margin:3px 0 12px;font-size:11px;color:var(--ink2)}
.viz .tiles{display:flex;flex-wrap:wrap;gap:10px;align-items:stretch}
.viz .tile{flex:1 1 130px;border:1px solid var(--rule);border-radius:5px;padding:9px 11px}
.viz .tile .k{font-size:10px;letter-spacing:.06em;text-transform:uppercase;color:var(--ink2)}
.viz .tile .v{font-size:19px;font-weight:600;color:var(--ink);margin-top:3px;
font-variant-numeric:tabular-nums}
.viz .tile .u{font-size:10px;color:var(--ink2);margin-top:1px}
.viz .op{flex:0 0 auto;align-self:center;font-size:16px;color:var(--ink2);padding:0 1px}
.viz .cols{display:flex;gap:12px;flex-wrap:wrap}
.viz .col{flex:1 1 270px;min-width:250px}
.viz .col h5{margin:0 0 5px;font-size:11px;font-weight:600;color:var(--ink)}
.viz .col h5 span{font-weight:400;color:var(--ink2)}
.viz .txt{border:1px solid var(--rule);border-radius:5px;padding:9px 10px;
font:11px/1.55 ui-monospace,SFMono-Regular,Menlo,monospace;color:var(--ink);
white-space:pre-wrap;max-height:330px;overflow:auto}
.viz mark{background:rgba(235,104,52,.22);color:var(--ink);border-radius:2px;padding:0 1px}
.viz .chips{margin:6px 0 0;line-height:1.9}
.viz .chip{display:inline-block;border:1px solid var(--rule);border-radius:3px;
padding:1px 6px;margin-right:4px;font-size:10px;color:var(--ink2)}
.viz .chip.gone{border-color:var(--c3);color:var(--c3)}
.viz .chip.left{border-color:var(--crit);color:var(--crit)}
.viz .warn{border:1px solid var(--crit);color:var(--crit);border-radius:4px;
padding:7px 9px;margin:0 0 10px;font-size:11px;font-weight:600}
</style>"""

_BAR_H, _BAND_H, _PLOT_W, _VAL_W, _CH, _LBL_MAX = 14, 26, 420, 78, 5.7, 268


def _esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _wrap(title, sub, body, footer=""):
    return HTML(_CSS + f'<div class="viz"><h4>{_esc(title)}</h4>'
                + (f'<p class="sub">{sub}</p>' if sub else "") + body
                + (f'<p class="sub" style="margin:9px 0 0">{footer}</p>' if footer else "")
                + "</div>")


def hbar(rows, title, sub="", *, fmt="{:.0f}", max_value=None, footer=""):
    """Horizontal bars: rows of (label, value, hue_1_to_4_or_None, hover_note)."""
    rows = [r for r in rows if r[1] is not None]
    if not rows:
        return _wrap(title, "no values available", "")
    top = max_value or max(abs(v) for _, v, _, _ in rows) or 1.0
    lbl_w = min(_LBL_MAX, max(70, int(max(len(str(r[0])) for r in rows) * _CH) + 6))
    x0, h = lbl_w + 10, _BAND_H * len(rows) + 6
    out = [f'<svg width="{x0 + _PLOT_W + _VAL_W}" height="{h}" role="img">',
           f'<line x1="{x0}" y1="2" x2="{x0}" y2="{h - 4}" stroke="var(--rule)"/>']
    for i, (label, value, hue, note) in enumerate(rows):
        y = i * _BAND_H + 3
        w = max(2.0, abs(value) / top * _PLOT_W)
        c = f"var(--c{hue or 0})"
        out += [f'<g><title>{_esc(label)}: {fmt.format(value)}{_esc(note)}</title>',
                f'<text x="{lbl_w}" y="{y + 11}" text-anchor="end" fill="var(--ink2)">'
                f'{_esc(label)}</text>',
                f'<rect x="{x0}" y="{y}" width="{w:.1f}" height="{_BAR_H}" rx="4" fill="{c}"/>',
                f'<rect x="{x0}" y="{y}" width="{min(4.0, w):.1f}" height="{_BAR_H}" fill="{c}"/>',
                f'<text x="{x0 + w + 7:.1f}" y="{y + 11}" fill="var(--ink)" '
                f'style="font-variant-numeric:tabular-nums">{fmt.format(value)}</text></g>']
    out.append("</svg>")
    return _wrap(title, sub, "".join(out), footer)


def tiles(items, title, sub=""):
    body = []
    for it in items:
        if it in ("x", "=", "->"):
            body.append('<div class="op">'
                        + {"x": "&times;", "=": "=", "->": "&rarr;"}[it] + "</div>")
        else:
            k, v, u = it
            body.append(f'<div class="tile"><div class="k">{_esc(k)}</div>'
                        f'<div class="v">{_esc(v)}</div><div class="u">{_esc(u)}</div></div>')
    return _wrap(title, sub, '<div class="tiles">' + "".join(body) + "</div>")


# The finite set of phrases the gazetteer substitutes in. Highlighting these is
# exact rather than heuristic: if a phrase is here, a pattern put it here.
_ROLE_PHRASES = sorted(set(gazetteer.ROLES.values()), key=len, reverse=True)


def _highlight_roles(text):
    """Mark every gazetteer role phrase in an escaped string."""
    out = _esc(text)
    for phrase in _ROLE_PHRASES:
        out = out.replace(_esc(phrase), f"<mark>{_esc(phrase)}</mark>")
    return out


def _proper_nouns(text):
    """Capitalised words that are not sentence-initial — a rough name detector.

    ponytail: heuristic, and only ever used to LABEL what a stage removed, never
    to decide anything. `assert_clean` is the thing that decides, and it uses the
    gazetteer's own patterns.
    """
    words, found = text.replace("\n", " ").split(" "), []
    for prev, word in zip(["."] + words, words):
        bare = word.strip(".,;:!?()\u201c\u201d\"'\u2019")
        if (bare[:1].isupper() and len(bare) > 2 and not prev.endswith((".", "!", "?"))
                and not bare.isupper()):
            found.append(bare)
    return sorted(set(found))


def layers(stages, title, sub="", footer=""):
    """Side-by-side text columns: (heading, note, text, highlight_roles).

    Each column after the first lists the proper nouns the previous column still
    had and this one does not — which layer caught what, at a glance.
    """
    cols, prev = [], None
    for heading, note, text, mark in stages:
        body = _highlight_roles(text) if mark else _esc(text)
        chips = ""
        if prev is not None:
            gone = sorted(set(_proper_nouns(prev)) - set(_proper_nouns(text)))
            chips = ('<div class="chips">' + "".join(
                f'<span class="chip gone">&minus; {_esc(g)}</span>' for g in gone)
                + "</div>") if gone else ""
        cols.append(f'<div class="col"><h5>{_esc(heading)} <span>{_esc(note)}</span></h5>'
                    f'<div class="txt">{body}</div>{chips}</div>')
        prev = text
    return _wrap(title, sub, '<div class="cols">' + "".join(cols) + "</div>", footer)


def scanned(text, roster, title, sub=""):
    """The integrity scan's own answer, rendered as chips."""
    hits = sorted(set(gazetteer.scan(text, roster)))
    body = ('<div class="chips">' + "".join(
        f'<span class="chip left">{_esc(h)}</span>' for h in hits) + "</div>"
        ) if hits else '<div class="chips"><span class="chip gone">clean</span></div>'
    return _wrap(title, sub, body)


def warn(message):
    return HTML(_CSS + f'<div class="viz"><div class="warn">{_esc(message)}</div></div>')


print("display kit ready - inline SVG/HTML, no plotting dependency")

## Pick a country, an anchor, and how much to spend

`AS_OF` is a **Monday** — `config.CADENCE` is `W-MON`, so the whole backfill is
weekly Mondays and the pilot is a scale model of the 48-country run rather than a
different experiment.

`RUN_MODEL_PASSES` is **off by default**. On, it makes three real OpenAI calls
(body rewrite, digest sweep, identifiability probe) on one article and prints
what they cost. Off, it shows the committed output of one real run instead, so
the before/after below is genuine either way.

In [ ]:
# ============================================================================
# THE RESOLUTION CELL — the one place the data source is decided.
#
# Everything below reads DATA_MODE and TAG and never re-checks. A notebook that
# is half real and half fixture is worse than either, because it looks complete
# and isn't.
# ============================================================================

ISO2 = "TR"                       # <-- change country here
AS_OF = datetime.date(2018, 11, 5)   # a Monday; config.CADENCE is W-MON
RUN_MODEL_PASSES = False          # True = three real API calls, cost printed

NAME = config.country_name(ISO2)
WINDOW_START, WINDOW_END = snapshot_select.window(AS_OF)

# A real window holds tens of articles. Fewer than this is a half-finished
# harvest, and topping it up from fixtures would produce a walkthrough that
# silently mixes real rows with invented ones — so it raises instead.
_MIN_STORE_ROWS = 8

# --- TEMPORARY SCAFFOLDING --------------------------------------------------
# Three synthetic rows in store.read_window() shape, one per branch of
# snapshot_select.usable_body(): a body that survives, a row that never had one,
# and a body captured after the anchor. The people, party, bank and law in the
# text are INVENTED — the point is that the gazetteer cannot know them, and that
# is true of invented names and real ones alike.
#
# DELETE THIS BLOCK once the one-country one-year harvest lands. It exists only
# so the notebook runs against an empty store.
_STAND_IN_ROWS = [
    {"url": "https://example.invalid/tr-lira-record-low",
     "publisher_link": "https://example.invalid/tr-lira-record-low",
     "title": "Turkish lira slides to record low as central bank holds rates at 50%",
     "abstract": "Turkey's currency hit \u20ba34.2 against the dollar after the "
                 "central bank left its policy rate unchanged at 50%.",
     "body": (
         "The Turkish lira fell to a record low of \u20ba34.2 against the dollar on "
         "Tuesday, extending a slide that has cost the currency 18% since January, "
         "after the Central Bank of the Republic of T\u00fcrkiye left its policy rate "
         "unchanged at 50%.\n\n"
         "Finance minister Kerem Ata\u011flu told reporters in Ankara that the "
         "government would not be pushed into an emergency move, adding that the "
         "Yeni Yol Partisi-led coalition remained committed to the disinflation "
         "programme it announced under the National Savings Act last spring.\n\n"
         "TurkStat put annual inflation at 61.8% in September, down from 75.4% a "
         "year earlier. Economists at Bo\u011faz Yat\u0131r\u0131m Bank, one of the largest "
         "domestic lenders, said the figure understated pressure on households in "
         "Istanbul and across Anatolia, where rents have risen faster than the "
         "headline index.\n\n"
         "Foreign investors have been net sellers of Turkish assets for six "
         "consecutive weeks. Holdings fell by \u20ac1.4bn, with German and US funds "
         "accounting for most of the outflow, according to CBRT data. A "
         "Frankfurt-based strategist compared the episode to the 2018 currency "
         "crisis, when the lira lost 28% in a single quarter."),
     "body_status": "recovered", "body_vintage": "api-native",
     "source_system": "guardian", "tier": "full",
     "published_at": datetime.datetime(2018, 10, 30, 6, 0, tzinfo=datetime.timezone.utc),
     "themes": ["order", "friction"]},
    {"url": "https://example.invalid/tr-households-inflation",
     "publisher_link": "https://example.invalid/tr-households-inflation",
     "title": "Turkey's Inflation Eases but Households Feel Little Relief",
     "abstract": "Annual price growth in Turkey slowed for a third month, but "
                 "rents and food costs in Istanbul continue to outpace the "
                 "official index. Economists say the disinflation is uneven.",
     "body": None, "body_status": "none", "body_vintage": None,
     "source_system": "nyt", "tier": "abstract-only",
     "published_at": datetime.datetime(2018, 10, 22, 11, 30, tzinfo=datetime.timezone.utc),
     "themes": ["order"]},
    {"url": "https://example.invalid/tr-press-freedom-trial",
     "publisher_link": "https://example.invalid/tr-press-freedom-trial",
     "title": "Turkish journalists face retrial as press-freedom rankings slide",
     "abstract": "Four reporters go back before an Ankara court this week.",
     "body": ("Four Turkish journalists returned to an Ankara courtroom on Monday, "
              "in a retrial that press-freedom groups say will test the "
              "independence of the judiciary. The case has become a reference "
              "point for investors weighing institutional risk in Turkey."),
     "body_status": "recovered", "body_vintage": "wayback-20181212",
     "source_system": "guardian", "tier": "full",
     "published_at": datetime.datetime(2018, 10, 11, 9, 15, tzinfo=datetime.timezone.utc),
     "themes": ["information"]},
]
# --- END TEMPORARY SCAFFOLDING ---------------------------------------------

try:
    rows = store.read_window(ISO2, WINDOW_START, WINDOW_END)
    store_error = None
except Exception as exc:
    rows, store_error = [], f"{type(exc).__name__}: {exc}"

n_store = len(rows)
if 0 < n_store < _MIN_STORE_ROWS:
    raise RuntimeError(
        f"{ISO2} {AS_OF}: the store holds {n_store} article(s) in "
        f"[{WINDOW_START:%Y-%m-%d}, {WINDOW_END:%Y-%m-%d}), fewer than the "
        f"{_MIN_STORE_ROWS} this walkthrough treats as a real window. That is a "
        f"half-finished harvest, not a thin week. Finish the harvest "
        f"(`python -m backend.util.pilot.run guardian --country {ISO2}`) or move "
        f"AS_OF. Refusing to top up from stand-ins: a walkthrough that mixes "
        f"real rows with invented ones looks complete and is not."
    )

DATA_MODE = "STORE" if n_store else "STAND-IN"
TAG = "" if DATA_MODE == "STORE" else "  [STAND-IN]"
if DATA_MODE == "STAND-IN":
    rows = _STAND_IN_ROWS

print("=" * 78)
print(f"  country          {NAME} ({ISO2})")
print(f"  anchor (as_of)   {AS_OF}  ({AS_OF:%A})")
print(f"  window           [{WINDOW_START:%Y-%m-%d %H:%M}Z, {WINDOW_END:%Y-%m-%d %H:%M}Z)"
      f"   {config.SNAPSHOT_WINDOW_DAYS}d, upper bound strict")
print(f"  store returned   {n_store} row(s)"
      + (f"   (read failed: {store_error})" if store_error else ""))
print(f"  DATA MODE        {DATA_MODE}   using {len(rows)} row(s)")
print(f"  model passes     {'LIVE - this notebook will spend money' if RUN_MODEL_PASSES else 'OFF - showing a committed capture'}")
print("=" * 78)
if DATA_MODE == "STAND-IN":
    display(warn(
        "STAND-IN DATA. The store holds no articles for this country and window, "
        "so every table and chart below is built from three synthetic rows. They "
        "exercise the real code paths but they are not real articles. Every "
        "heading below carries a [STAND-IN] marker."))

## 1 — The no-future line

`snapshot_select` is the seam: the only difference between a historical run and a
live one, and therefore the only place hindsight can enter. Three rules, in order
of how badly each one bites.

**The window is strict at the top.** `[as_of − 30d, as_of)`. An article published
*on* the anchor is same-day news the live run's own `now() − 30d` cutoff would not
reliably have had either.

**A body may not be younger than the anchor.** An article published in June and
captured by the Wayback Machine in August is a June article with an *August body*
— publishers edit, append and re-headline. When the capture is younger than the
anchor the body is dropped and the article stays as title and abstract: thinner
evidence, honestly thin, rather than richer evidence that is a lie.

**A live refetch enters only if the leakage scan cleared it.** A page fetched
today is younger than any historical anchor by construction, so the only thing
making it usable is the step-4 scan that asks the model whether the text knows
anything that happened after publication — and fails closed.

Nothing here drops an *article*. Everything it refuses still reaches the scorer as
title and abstract.

In [ ]:
verdicts = []
for row in rows:
    vintage = row.get("body_vintage")
    captured = snapshot_select.capture_date(vintage)
    kept = snapshot_select.usable_body(row, AS_OF)
    if not row.get("body"):
        why = "never had one — the archive returns no body for this tier"
    elif kept:
        why = ("came with the article" if vintage == "api-native"
               else "leakage scan cleared it" if vintage == "live-refetch"
               else f"captured {captured} < anchor")
    else:
        why = (f"captured {captured} >= anchor — that is a later edition of the page"
               if captured else f"unreadable vintage {vintage!r}")
    verdicts.append({
        "title": (row["title"] or "")[:52],
        "source": row.get("source_system"), "tier": row.get("tier"),
        "published": row["published_at"].date(),
        "vintage": vintage, "body_chars": len(row.get("body") or ""),
        "body_kept": bool(kept), "why": why,
    })

print(f"Vintage verdicts — {NAME} as of {AS_OF}{TAG}")
display(pd.DataFrame(verdicts).set_index("title"))

# The canonical items the live pipeline consumes. Same normalize_item() the
# Google News path calls, so nothing downstream can tell where they came from.
items = [snapshot_select.to_item(row, AS_OF, NAME) for row in rows]
dropped = sum(1 for v in verdicts if v["body_chars"] and not v["body_kept"])
print(f"\n{len(items)} canonical item(s); {dropped} body/ies dropped as hindsight, "
      f"{sum(1 for v in verdicts if not v['body_chars'])} never had one.")
print("Those articles are NOT dropped — they reach the scorer as title + abstract.")

## 2 — Selection, on the live rules

The historical counterpart of `article_enrichment.fetch_relevant_news`, and
deliberately the same shape: score, threshold, abstract cap, per-theme floor.
Selection is not re-implemented — the relevance function, the floor and the
threshold constants are all imported from the live path, because two copies of
"the 20 articles" would be a silent disagreement about what the historical series
is comparable to.

Two rules worth knowing:

**The threshold orders the pool, it does not cap it.** Articles over the bar come
first, then the rest by rank until the budget is full. Read as a cap it produced
three-article weeks next to twenty-article ones for a one-article difference in
how many cleared 0.3.

**The abstract tier is rationed.** The NYT archive returns no bodies and is mostly
about the United States, so left uncapped a US snapshot fills with headlines while
a Portugal one keeps full bodies — and the two stop being the same instrument
pointed at different countries.

In [ ]:
MAX_ARTICLES = 20     # the same budget the live run spends

for item in items:
    item["relevance_score"] = article_ranking.score_relevance(item, NAME)

pool = sorted(items, key=lambda i: -i["relevance_score"])
clearing = [i for i in pool if i["relevance_score"] >= article_enrichment._RELEVANCE_THRESHOLD]
topped = clearing
if len(clearing) < MAX_ARTICLES:
    seen = {id(i) for i in clearing}
    topped = clearing + [i for i in pool if id(i) not in seen]
rationed = snapshot_select.ration_abstracts(topped, MAX_ARTICLES)
selected = core.select_with_theme_floor(rationed, MAX_ARTICLES,
                                        article_enrichment._PER_THEME_FLOOR)

display(hbar(
    [("in window", len(rows), None, "  — store.read_window, upper bound strict"),
     (f"cleared {article_enrichment._RELEVANCE_THRESHOLD}", len(clearing), 3,
      "  — the rest are ranked in behind them, not discarded"),
     ("after abstract cap", len(rationed), 4,
      f"  — at most {int(MAX_ARTICLES * config.ABSTRACT_TIER_SHARE)} abstract-only rows"),
     ("selected", len(selected), 1,
      f"  — per-theme floor {article_enrichment._PER_THEME_FLOOR}, budget {MAX_ARTICLES}")],
    f"The selection funnel — {NAME} as of {AS_OF}{TAG}",
    "Empty is a legitimate answer for a thin week. Inventing articles to fill a "
    "quota is the failure this whole machine exists to avoid."))

# The stable ids the scorer cites back.
for i, it in enumerate(selected, start=1):
    it["id"] = f"a{i}"

# Proof this is the real path and not a re-implementation. A no-op under
# stand-ins; the moment a harvest exists it checks every step above at once.
if DATA_MODE == "STORE":
    assert [i["link"] for i in selected] == [
        i["link"] for i in snapshot_select.select(ISO2, AS_OF, MAX_ARTICLES)], \
        "the cells above disagree with snapshot_select.select — one of them is wrong"
    print("checked: these cells reproduce snapshot_select.select() exactly")
else:
    print("skipped the snapshot_select.select() cross-check: nothing in the store to select from")

display(pd.DataFrame([
    {"id": i["id"], "rel": round(i["relevance_score"], 3), "theme": i.get("_theme"),
     "tier": i.get("tier"), "body": len(i.get("text") or ""),
     "title": (i["title"] or "")[:56]} for i in selected]).set_index("id"))

## 3 — Layer 1: the gazetteer

Deterministic, offline, no model, no network. A list of every surface form that
identifies a roster country, mapped to the **functional role it plays** — chosen
so three things survive:

- **Numbers.** "inflation hit 61.8%" stays "inflation hit 61.8%". A masked run
  that also lost the magnitudes would be measuring something else entirely.
- **Roles.** "the Central Bank of the Republic of Türkiye" becomes "the central
  bank", not a hole. The scorer still needs to know a central bank did the thing.
- **Coarse region.** A country becomes "the country"; its neighbours become "a
  neighbouring country". Turning "North Korea" into "the country" would be both
  wrong and a giveaway.

Two passes, in this order, and the order is load-bearing: `mask()` first, so the
scored country's central bank survives as "the central bank"; then
`mask_foreign()`, so every *other* roster country collapses to "another country".
Running the flat pass first would eat the specific one.

**What is deliberately left alone.** "real" and "won" are ordinary English far
more often than they are money — masking them turns "real GDP" and "the party won
the election" into nonsense. Same for the ambiguous ISO codes (`PEN`, `COP`,
`PHP`, `SAR`, `CAD`) and for region names that are ordinary phrases ("the South",
"the West"). A corpus that reads as *damaged* tells the scorer something was
removed, which is a worse leak than the word it was hiding.

In [ ]:
terms = gazetteer.terms(ISO2)
tier = "curated" if ISO2 in ("US", "TR", "BR", "PT", "KR") else "thin (roster + babel)"
print(f"{ISO2} gazetteer entry: {len(terms)} surface forms, {tier} tier")
print("  longest first, so 'Central Bank of the Republic of Türkiye' is consumed "
      "before 'Türkiye':")
for form in terms[:6]:
    print(f"    {form}")
print(f"    ... and {len(terms) - 6} more")
print(f"\nleft alone by design: {', '.join(gazetteer.UNMASKED_BY_DESIGN)}  "
      f"(+ ambiguous ISO codes {', '.join(gazetteer.AMBIGUOUS_CODES)})")

# One article's body through both passes. mask_text() is the two-pass wrapper.
subject = next((i for i in selected if i.get("text")), selected[0])
raw = subject.get("text") or subject["title"]
own_only = gazetteer.mask(raw, ISO2)                    # pass 1 alone
both = rewrite.mask_text(raw, ISO2)                     # pass 1 then pass 2

display(layers(
    [("raw", "as harvested", raw, False),
     ("+ mask()", "the scored country by its roles", own_only, True),
     ("+ mask_foreign()", "every other roster country flattened", both, True)],
    f"Layer 1, both passes — {subject['id']}{TAG}",
    "Highlighted phrases are gazetteer role substitutions. The chips under each "
    "column are the proper nouns the previous column had and this one does not.",
    footer="Note what survived: every number, and every institutional role."))

print("\nTwo artifacts worth seeing rather than glossing over, both visible above:")
print("  'the the central bank'  — mask() replaces the institution's name but leaves")
print("     the article in front of it. mask_foreign() swallows an optional leading")
print("     'the' precisely to avoid this; the scored-country pass does not, so the")
print("     country's OWN institutions read worse than the foreign ones.")
print("  'the local currency 34.2' after 'the local currency' — 'lira' and '₺' are")
print("     two forms of the same currency sitting next to each other, and each is")
print("     substituted independently.")
print("Neither leaks. Both make the text read as damaged, which the module docstrings")
print("call a leak of a different kind — and both are what layer 2 tidies up.")

display(scanned(both, [ISO2], f"Integrity scan of that body against {ISO2}{TAG}",
                "gazetteer.scan() — the same function assert_clean() uses. "
                "Clean here means the LIST found nothing; it does not mean the "
                "text is unidentifiable. Layer 2 and the probe are what test that."))

In [ ]:
# The whole bundle, field by field. mask_item() skips ids and links and masks
# everything else — a gate defaults to masking, and what it skips has to be
# argued for. The links are argued for: they are never sent, and a path like
# ".../2018/aug/13/turkey-lira-crisis" masks into nonsense.
masked_items = rewrite.mask_items(selected, ISO2)

before = [len(set(gazetteer.scan(json.dumps(i, default=str), gazetteer.DEFAULT_ROSTER)))
          for i in selected]
after = [len(set(gazetteer.scan(json.dumps({k: v for k, v in i.items()
                                            if k not in ("link", "publisher_link")},
                                           default=str), gazetteer.DEFAULT_ROSTER)))
         for i in masked_items]

display(hbar(
    [(f"{i['id']} before", b, 2, "  — roster forms in the stored item")
     for i, b in zip(selected, before)]
    + [(f"{i['id']} after", a, 3, "  — roster forms after layer 1")
       for i, a in zip(selected, after)],
    f"Roster forms per article, whole roster not just {ISO2}{TAG}",
    "Scanned against all 48 countries: an article naming a DIFFERENT roster "
    "country lets the probe rule countries out by elimination, which is the same "
    "leak wearing a hat.",
    footer="Links are excluded from the 'after' scan — they are never sent."))

# The evidence payload names the country in its _meta too, and it is serialized
# whole into the prompt. mask_payload walks keys as well as values.
demo_payload = {"_meta": {"country": ISO2, "country_name": NAME,
                          "generated_at": AS_OF.isoformat()},
                "series": {f"Exchange rate vs USD": {"value": 5.4, "unit": "TRY/USD"}}}
print("evidence payload, masked whole — keys as well as values:")
print(json.dumps(rewrite.mask_payload(demo_payload, ISO2), ensure_ascii=False, indent=2))
print("\n_meta.country was the exact string 'TR' — a two-letter code is catastrophic as "
      "a prose pattern ('IT', 'NO', 'IN' are ordinary words) but a payload VALUE that "
      "IS a code is the loudest possible leak, so it is matched on the whole string.")

## 4 — Layer 2: what a list cannot know

Ten years of news is ten years of politicians, parties, companies, laws and
stadiums, and no hand-written list survives that. The gazetteer masks what
somebody wrote down; it does not know this year's finance minister.

So a model is asked to replace what remains with the role it plays — a named
finance minister becomes "the finance minister" — under one non-negotiable
instruction: **keep every number exactly as written.**

Two passes with two different failure modes, and the asymmetry is deliberate:

| | Runs on | On failure |
|---|---|---|
| `sweep_digest` | every article's digest + its headline | **fails open** — keeps the unswept digest, because a digest is not sent whole and dropping it silently would cost the article |
| `rewrite_body` | the 2–3 bodies the scorer reads end to end | **fails closed** — returns `""`, the article degrades to its masked title. Being short one body costs a week some evidence; one leaked name costs the whole comparison |

Rules 3 and 5 of the shared prompt exist because a probe measured them: with
people and countries gone, six of six bundles were still identified at 0.80–0.90,
citing "the Help America Vote Act", "the White House Situation Room", "as bad as
Brexit". A statute, a building and an event — none of them reachable from a list
of country names, and extending the list would mean enumerating every proper noun
on earth. This is the layer that generalises, so this is where the scope belongs.

Both passes are cached on a hash of the **masked** text, which is what makes a
backfill reproducible: `input_manifest` hashes the bytes the model read, and for
these articles those bytes are generated prose kept nowhere else.

In [ ]:
# One real run, committed so the offline path shows a genuine before/after rather
# than prompts. CAPTURED-NOT-LIVE. Re-capture when any mask version below moves.
CAPTURED = {'captured_on': '2026-08-14',
 'rewritten': 'The local currency fell to a record low of the local currency 34.2 '
              'against another country on Tuesday, extending a slide that has cost '
              'the currency 18% since January, after the central bank left its '
              'policy rate unchanged at 50%.\n'
              '\n'
              'The finance minister told reporters in the capital that the '
              'government would not be pushed into an emergency move, adding that '
              'the governing coalition remained committed to the disinflation '
              'programme it announced under a national savings law last spring.\n'
              '\n'
              'The national statistics office put annual inflation at 61.8% in '
              'September, down from 75.4% a year earlier. Economists at a large '
              'domestic bank, one of the largest domestic lenders, said the figure '
              'understated pressure on households in a major city and across the '
              'region, where rents have risen faster than the headline index.\n'
              '\n'
              "Foreign investors have been net sellers of the country's assets for "
              'six consecutive weeks. Holdings fell by another country 1.4bn, with '
              'another country and another country funds accounting for most of the '
              'outflow, according to the central bank data. A strategist based in a '
              'major foreign city compared the episode to a previous currency '
              'crisis, when the local currency lost 28% in a single quarter.',
 'swept': {'what_happened': 'The central bank held its policy rate at 50% and the '
                            'local currency fell to a record low of 34.2 against '
                            'another country.',
           'actors': 'the finance minister, the governing party-led coalition, the '
                     'central bank, and economists at a large domestic bank.',
           'numbers': 'Policy rate 50%; the local currency 34.2/another country, '
                      'down 18% since January; the national statistics office '
                      'inflation 61.8% in September against 75.4% a year earlier; '
                      'another country 1.4bn of outflows over six weeks.',
           'transmission': 'Currency weakness feeds import prices and household '
                           'rents in a major city and across the region while '
                           "foreign holdings of the country's assets fall.",
           'stage1_severity': 4,
           'masked_title': 'the local currency slides to record low as central bank '
                           'holds rates at 50%'},
 'probe_layer1': {'country': 'TR',
                  'confidence': 0.85,
                  'alternatives': [{'country': 'TR', 'probability': 0.85},
                                   {'country': 'ZZ', 'probability': 0.1},
                                   {'country': 'CY', 'probability': 0.05}],
                  'insufficient_information': False,
                  'evidence': 'The mention of a central bank holding a policy rate '
                              'at 50%, a local currency sliding to a record low, and '
                              "the specific names like 'Yeni Yol Partisi' and 'Kerem "
                              "Atağlu' strongly indicate Turkey."},
 'probe_layer2': {'country': 'ZZ',
                  'confidence': 0.0,
                  'alternatives': [{'country': 'AR', 'probability': 0.4},
                                   {'country': 'TU', 'probability': 0.4},
                                   {'country': 'VE', 'probability': 0.2}],
                  'insufficient_information': True,
                  'evidence': 'The text mentions a high policy rate of 50%, '
                              'significant currency depreciation, and high inflation '
                              'rates, which are characteristics seen in several '
                              'countries experiencing economic crises, but does not '
                              'provide specific identifiers.'},
 'probe_null': {'country': 'ZZ',
                'confidence': 0.0,
                'alternatives': [{'country': 'ZZ', 'probability': 1.0},
                                 {'country': 'ZZ', 'probability': 0.0},
                                 {'country': 'ZZ', 'probability': 0.0}],
                'insufficient_information': True,
                'evidence': 'The summaries contain general economic indicators and '
                            'events without specific country identifiers.'},
 'spend_usd': 0.001,
 'versions': {'mask_map_version': 'g5',
              'gazetteer_version': 'aa63700b',
              'sweep_version': 'be4649c6',
              'rewrite_version': '03de44ac',
              'probe_version': 'd089c696',
              'digest_model': 'gpt-4o-mini-2024-07-18'}}

_LIVE_VERSIONS = {"mask_map_version": gazetteer.MASK_MAP_VERSION,
                  "gazetteer_version": gazetteer.GAZETTEER_VERSION,
                  "sweep_version": rewrite.SWEEP_VERSION,
                  "rewrite_version": rewrite.REWRITE_VERSION,
                  "probe_version": probe.PROBE_VERSION,
                  "digest_model": ai_client.DIGEST_MODEL_NAME}
_DRIFTED = {k: (v, _LIVE_VERSIONS[k]) for k, v in CAPTURED["versions"].items()
            if _LIVE_VERSIONS[k] != v}

# The body layer 1 already masked, and the digest as the stage-1 model wrote it.
gaz_body = rewrite.mask_text(subject.get("text") or "", ISO2)
gaz_title = rewrite.mask_text(subject["title"], ISO2)
demo_digest = rewrite.mask_payload(
    subject.get("digest") or {
        "what_happened": "The Central Bank of the Republic of T\u00fcrkiye held its "
                         "policy rate at 50% and the lira fell to a record \u20ba34.2.",
        "actors": "Finance minister Kerem Ata\u011flu, the Yeni Yol Partisi-led "
                  "coalition, the CBRT, and economists at Bo\u011faz Yat\u0131r\u0131m Bank.",
        "numbers": "Policy rate 50%; lira \u20ba34.2/USD, down 18% since January; "
                   "TurkStat inflation 61.8% in September against 75.4%.",
        "transmission": "Currency weakness feeds import prices and household rents.",
    }, ISO2)

if RUN_MODEL_PASSES:
    api_key = os.environ["OPENAI_API_KEY"]
    with usage.meter() as meter:
        model_body = rewrite.rewrite_body(gaz_body, api_key)
        swept = rewrite.sweep_digest(demo_digest, api_key, title=gaz_title)
    source_note = "LIVE"
    print(f"metered spend for the two passes above: ${meter.spend_usd:.4f}")
    if not model_body:
        print("the rewrite failed or came back empty — FAIL CLOSED: this article "
              "would reach the scorer as its masked title and nothing else.")
else:
    model_body, swept = CAPTURED["rewritten"], CAPTURED["swept"]
    source_note = f"CAPTURED {CAPTURED['captured_on']}, not live"
    if _DRIFTED:
        display(warn("RE-CAPTURE NEEDED — the masking behaviour has changed since "
                     "this output was captured: "
                     + "; ".join(f"{k} {was} -> {now}" for k, (was, now) in _DRIFTED.items())
                     + ". The before/after below no longer describes what the code does. "
                       "Re-run this cell with RUN_MODEL_PASSES = True and paste the result "
                       "into CAPTURED."))
    else:
        print(f"offline: showing a real run captured {CAPTURED['captured_on']} "
              f"for ${CAPTURED['spend_usd']:.4f}. Every mask version still matches.")

display(layers(
    [("raw", "as harvested", subject.get("text") or subject["title"], False),
     ("layer 1", "gazetteer — a list", gaz_body, True),
     ("layer 2", f"model rewrite — {source_note}", model_body or "(empty: failed closed)", True)],
    f"Which layer caught what — {subject['id']}{TAG}",
    "The chips under layer 2 are exactly what a list could never have held: a "
    "person, a party, a bank, a named law.",
    footer="Every number is unchanged across all three columns. That is the one "
           "instruction the prompt calls non-negotiable."))

print("\ndigest sweep — the fields the stage-1 model wrote, swept of the names "
      "it was told in the same breath not to write:")
for field in ("what_happened", "actors", "numbers", "transmission"):
    print(f"\n  {field}")
    print(f"    before  {demo_digest.get(field, '')}")
    print(f"    after   {(swept or {}).get(field, '')}")
print(f"\n  headline")
print(f"    before  {gaz_title}")
print(f"    after   {(swept or {}).get('masked_title', '')}")
print("\nThe headline is swept in the same call as the digest, so it costs nothing "
      "extra\nand caches with it. It is sent for EVERY article, digest or not, and it "
      "was\nreaching the model on gazetteer masking alone: six of twenty titles in one "
      "\nmeasured bundle still named the politician.")

## 5 — The gate

The gazetteer is a list somebody wrote and the sweep is a model, so neither is
trusted. Before anything leaves for the API, `assert_clean` walks the **whole
outbound payload** — nested dicts, lists, dict *keys* as well as values — and
scans every string against the **whole roster**.

Keys matter because the payload is serialized to JSON before it reaches the
model, so a label is as visible as a number, and the evidence payload really does
carry `"Exchange rate vs USD"` as a key.

The whole roster matters because an article naming a *different* country lets the
probe rule countries out by elimination.

A hit raises `MaskLeak`. Not a warning: a masked snapshot that names its country
is not a degraded result, it is a wrong one, and it would sit in the series
looking exactly like a right one.

In [ ]:
# What production scans: the serialized blocks, not the objects they came from.
# Those objects carry fields the model never sees \u2014 the article URLs, which name
# the country in their paths and mask into nonsense if touched \u2014 so scanning them
# means either a gate that cries wolf on every snapshot or an allow-list of what
# to scan, which is the same allow-list that already let `content` and `summary`
# through. Not the whole prompt either: the template's own worked examples name
# Australia and China, and those are instructions rather than evidence.
entries = langchain_llm.prompt_entries(masked_items)
outbound = [json.dumps(rewrite.mask_payload(demo_payload, ISO2), ensure_ascii=False),
            langchain_llm._digests_to_json(masked_items),
            langchain_llm._fulltext_block(masked_items, [masked_items[0]["id"]])]
rewrite.assert_clean(outbound)
print(f"assert_clean passed on {len(outbound)} serialized block(s) carrying "
      f"{len(entries)} article(s){TAG}")
for block, label in zip(outbound, ("evidence_json", "articles_json", "full_text_block")):
    print(f"  {label:<17} {len(block):>7,} chars")

# The gate's own check. If this ever stops raising, the gate is broken and every
# masked row written afterwards is unverifiable.
leaky = list(outbound)
leaky[1] = leaky[1].replace("the country", "T\u00fcrkiye", 1)
leaky.append(json.dumps({"_meta": {"country": ISO2}}))
try:
    rewrite.assert_clean(leaky)
    raise AssertionError("assert_clean did NOT raise on a payload naming its country")
except rewrite.MaskLeak as leak:
    print(f"\nput one role phrase back as the country's own name, and added a "
          f"_meta block naming its ISO2:\n  MaskLeak: {leak}")
print("\nThat is what stands between a mislabelled row and the series.")

## 6 — The meter, not a gate

`assert_clean` proves the *list* found nothing. It cannot prove the bundle is
unidentifiable — a wine region identifies a country as precisely as its central
bank does. A live masked run passed the integrity scan with zero flagged tokens
and the probe still named Portugal at 0.9, citing the Douro Valley and the
Algarve.

So the probe asks a cheap model, straight out: which country is this? The answer
is **recorded and never enforced**. It could not be — the US is expected to be
identified nearly always from coverage volume alone, and refusing to score the US
would be answering the wrong question. In production it runs on one country in
six, deterministically seeded on `(country, date)` so a re-run probes the same
countries.

**The control arm is what makes the number mean anything.** A probe forced to
name a country will name the one its prior favours, and on a roster containing
the United States that is the United States — so "US identified at 0.85" and "the
model always says US" produce identical output. `probe.null_bundle()` is a bundle
with no country in it at all, written by hand rather than stripped from real
articles, with plausible numbers kept.

**Read-only here.** `probe.probe()` is a pure function; the `probe_result` write
lives in `pipeline._identifiability`, which this notebook never calls. Keep it
that way — the identifiability baseline is the meter that validates masking, and
a teaching artifact must not contaminate it.

In [ ]:
def probe_row(guess, label):
    alts = ", ".join(f"{a['country']} {a['probability']:.2f}"
                     for a in guess.get("alternatives") or []) or "—"
    return {"bundle": label, "guessed": guess.get("country"),
            "confidence": round(guess.get("confidence", 0.0), 2),
            "no_guess": guess.get("insufficient_information"),
            "top-3": alts, "evidence": (guess.get("evidence") or "")[:90]}

layer1_item = {"id": "a1", "title": gaz_title, "digest": demo_digest, "text": gaz_body}
layer2_item = {"id": "a1", "title": (swept or {}).get("masked_title") or gaz_title,
               "digest": {k: v for k, v in (swept or {}).items() if k != "masked_title"},
               "text": model_body or ""}

if RUN_MODEL_PASSES:
    api_key = os.environ["OPENAI_API_KEY"]
    with usage.meter() as meter:
        g1 = probe.probe([layer1_item], api_key, fulltext_ids=["a1"])
        g2 = probe.probe([layer2_item], api_key, fulltext_ids=["a1"])
        gnull = probe.probe(probe.null_bundle(6), api_key)
    print(f"metered spend for three probes: ${meter.spend_usd:.4f}")
else:
    g1, g2, gnull = (CAPTURED["probe_layer1"], CAPTURED["probe_layer2"],
                     CAPTURED["probe_null"])
    print(f"offline: probes captured {CAPTURED['captured_on']}, not live")

print(f"\nIdentifiability — truth is {ISO2}{TAG}")
display(pd.DataFrame([
    probe_row(g1, "layer 1 only (gazetteer)"),
    probe_row(g2, "layers 1 + 2 (gazetteer + model)"),
    probe_row(gnull, "null bundle (control)"),
]).set_index("bundle"))

display(hbar(
    [("layer 1 only", g1.get("confidence", 0.0), 2, "  — the list alone"),
     ("layers 1 + 2", g2.get("confidence", 0.0), 3, "  — after the model pass"),
     ("null control", gnull.get("confidence", 0.0), None, "  — nothing to identify")],
    f"Probe confidence in its own guess{TAG}", fmt="{:.2f}", max_value=1.0,
    sub="Lower is better, and the control is the floor. This is a measurement, "
        "never a gate: a confident correct guess does not stop the snapshot.",
    footer="What the probe quotes back is the useful part — it names the exact "
           "text that gave the country away."))

print(f"\nlayer 1 evidence: {g1.get('evidence')}")
print(f"layer 2 evidence: {g2.get('evidence')}")
print("\nNothing was written to probe_result: probe.probe() is a pure function, and "
      "the writer lives in pipeline._identifiability, which this notebook never calls.")

## 7 — What gets stamped, and where the row lands

A masked row records the **full masking behaviour that produced it**, because a
score you cannot reproduce is a score you cannot defend. Five version fields,
each earned by a specific failure:

- `mask_map_version` — hand-maintained, versions the map's *data*. Its limit is
  that a human has to remember to bump it.
- `gazetteer_version` — sha256 of `gazetteer.py`. The euro fix changed masking
  *code* and moved neither the data nor its version. A hash cannot forget.
- `sweep_version` — sha256 of the sweep prompt. The digest cache keys on a hash
  of the masked *text*, and the sweep runs after the digest is generated, so the
  sweep changed twice while the cache key sat still: two masking behaviours, one
  cache key.
- `rewrite_version` — sha256 of the body prompt. Separate from the sweep because
  the body rewrite cannot change a digest; folding them together threw away every
  cached digest whenever the body prompt moved.
- `identifiability` — the probe's own answer, stored beside the row.

And three arms, of which **only one is production**:

| mode | writes to | why |
|---|---|---|
| `masked` | `risk_snapshot` | the continuous weekly series; the regime that becomes production |
| `named` | `history_run_ledger` | the diagnostic twin — what identity was worth |
| `masked_nostructural` | `history_run_ledger` | masked, `structural` withheld. Masked-vs-named divergence is ambiguous alone: a small gap could mean the structural facts recovered what the name carried, or that the name never mattered. Only the third arm separates those. |

The two diagnostic arms share `(country, as_of)` with their masked twin and would
overwrite the production series on its own primary key. A series that silently
changes scoring regime half way through its own history is worse than no series.

In [ ]:
# Assembled exactly as pipeline._process_country does, minus the write.
masking_block = {
    "scoring_mode": "masked",
    "mask_map_version": gazetteer.MASK_MAP_VERSION,
    "gazetteer_version": gazetteer.GAZETTEER_VERSION,
    "sweep_version": rewrite.SWEEP_VERSION,
    # "clean" by construction: country_llm_score raises MaskLeak before sending,
    # so any row that exists at all got past the gate. Recorded anyway, because a
    # manifest that only says what went right when it went right proves nothing.
    "mask_integrity_status": "clean",
    "structural_fields": 0,
    "identifiability": g2,
}
print(f"input_manifest.masking — {NAME} {AS_OF}{TAG}")
print(json.dumps(masking_block, ensure_ascii=False, indent=2))

display(pd.DataFrame([
    {"mode": m,
     "writes to": "history_run_ledger" if m in config.DIAGNOSTIC_MODES else "risk_snapshot",
     "production": m not in config.DIAGNOSTIC_MODES}
    for m in config.SCORING_MODES]).set_index("mode"))

## What this notebook did not do

It never scored and it never wrote. No `pipeline._process_country`, no
`data_push.upsert_snapshot`, no `data_push.upsert_probe_result`, no
`store.write_run`, no ledger row, no spend beyond the model passes above.

The real thing is the CLI, and every command that spends money prints its
projection and waits for a yes:

```
python -m backend.util.pilot.run guardian --country TR   # harvest
python -m backend.util.pilot.run wayback                 # recover bodies
python -m backend.util.pilot.run score --country TR --since 2018-01-01 --until 2018-12-31
python -m backend.util.pilot.run diagnostic              # the named arms
python -m backend.util.pilot.run pilot-report            # the five meters
```

Everything after `snapshot_select.select()` in that `score` command is
`pipeline._process_country` — the same function the daily run calls, with `as_of`
pinned. `backend/notebooks/country_rating_walkthrough.ipynb` walks that half.

In [ ]:
print(f"{NAME} ({ISO2})  anchor {AS_OF}  data {DATA_MODE}")
print(f"  window         [{WINDOW_START:%Y-%m-%d}, {WINDOW_END:%Y-%m-%d})  "
      f"{len(rows)} row(s) in, {len(selected)} selected")
print(f"  vintage        {dropped} body/ies refused as hindsight")
print(f"  layer 1        {len(gazetteer.terms(ISO2))} surface forms for {ISO2}, "
      f"{len(gazetteer.DEFAULT_ROSTER)} countries flattened as foreign")
print(f"  layer 2        {'live' if RUN_MODEL_PASSES else 'captured ' + CAPTURED['captured_on']}"
      f"  sweep {rewrite.SWEEP_VERSION}  rewrite {rewrite.REWRITE_VERSION}")
print(f"  gate           assert_clean passed; MaskLeak raised on the injected name")
print(f"  meter          layer 1 {g1.get('country')} @ {g1.get('confidence'):.2f}"
      f"  ->  layers 1+2 {g2.get('country')} @ {g2.get('confidence'):.2f}"
      f"  (control {gnull.get('country')} @ {gnull.get('confidence'):.2f})")
print(f"  wrote          nothing")